# Agent Sheep (Phase 3: MQTT Publisher)

This notebook runs the sheep-only simulation and publishes per-tick messages to MQTT topics.

Tick order stays the same: move → eat → reproduce → die → regrow grass.

It loads settings via `simulated_city.config.load_config()` and uses `MqttConnector` + `MqttPublisher` for publishing.

In [ ]:
# Cell purpose: import helpers, load config, and connect to MQTT broker.
from datetime import datetime, timezone
import json

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
from simulated_city.sheep_wolf_models import SheepSimulationParams, run_simulation

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-sheep-phase3")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)

base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
sheep_state_topic = f"{base_topic}/sim/sheep/state"
events_topic = f"{base_topic}/sim/events"

print("Publish topics:")
print(f"- {tick_topic}")
print(f"- {sheep_state_topic}")
print(f"- {events_topic}")

Loaded config. Primary MQTT profile: 127.0.0.1:1883


In [ ]:
# Cell purpose: define Phase 3 simulation parameters and deterministic seed.
simulation_cfg = config.simulation

params = SheepSimulationParams(
    grid_width=(simulation_cfg.grid_width if simulation_cfg is not None else 10),
    grid_height=(simulation_cfg.grid_height if simulation_cfg is not None else 10),
    initial_sheep=(simulation_cfg.initial_sheep if simulation_cfg is not None else 30),
    initial_sheep_energy=(simulation_cfg.sheep_initial_energy if simulation_cfg is not None else 8),
    sheep_move_cost=(simulation_cfg.sheep_move_cost if simulation_cfg is not None else 1),
    sheep_eat_gain=(simulation_cfg.sheep_eat_gain if simulation_cfg is not None else 4),
    sheep_reproduction_threshold=(
        simulation_cfg.sheep_reproduction_threshold if simulation_cfg is not None else 10
    ),
    sheep_reproduction_probability=(
        simulation_cfg.sheep_reproduction_probability if simulation_cfg is not None else 0.08
    ),
    grass_regrow_ticks=(simulation_cfg.grass_regrow_ticks if simulation_cfg is not None else 5),
)

seed = 42
if simulation_cfg is not None and simulation_cfg.seed is not None:
    seed = int(simulation_cfg.seed)

tick_count = 30
print(f"Using seed={seed}, ticks={tick_count}, grid={params.grid_width}x{params.grid_height}")

Using seed=42, ticks=30, grid=10x10


In [ ]:
# Cell purpose: run simulation and publish tick/state/event payloads to MQTT.
summaries = run_simulation(params=params, ticks=tick_count, seed=seed)

print("tick sheep grass births deaths ate avg_energy")
for summary in summaries:
    tick_payload = {
        "tick": summary.tick,
        "seed": seed,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    state_payload = {
        "tick": summary.tick,
        "seed": seed,
        "sheep_count": summary.sheep_count,
        "grass_grown_cells": summary.grass_grown_cells,
        "average_energy": round(summary.average_energy, 3),
    }
    event_payload = {
        "tick": summary.tick,
        "seed": seed,
        "births": summary.births,
        "deaths": summary.deaths,
        "sheep_ate_grass": summary.sheep_ate_grass,
    }

    publisher.publish_json(tick_topic, json.dumps(tick_payload), qos=0)
    publisher.publish_json(sheep_state_topic, json.dumps(state_payload), qos=0)
    publisher.publish_json(events_topic, json.dumps(event_payload), qos=0)

    print(
        f"{summary.tick:>4} {summary.sheep_count:>5} {summary.grass_grown_cells:>5} "
        f"{summary.births:>6} {summary.deaths:>6} {summary.sheep_ate_grass:>3} "
        f"{summary.average_energy:>10.2f} | published"
    )

final_summary = summaries[-1]
print()
print("Final summary:")
print(
    {
        "tick": final_summary.tick,
        "sheep_count": final_summary.sheep_count,
        "grass_grown_cells": final_summary.grass_grown_cells,
        "average_energy": round(final_summary.average_energy, 3),
    }
)

tick sheep grass births deaths ate avg_energy
   1    30    73      0      0  27      10.60
   2    33    50      3      0  23      11.52
   3    36    33      3      0  17      11.53
   4    39    27      3      0   6      10.33
   5    42    50      3      0   4       9.05
   6    42    54      0      0  19       9.86
   7    43    55      1      0  16      10.14
   8    46    48      3      0  13       9.67
   9    48    41      2      0  11       9.23
  10    49    54      1      0   6       8.55
  11    51    53      2      0  17       8.59
  12    51    47      1      1  19       9.08
  13    52    43      1      0  15       9.08
  14    51    33      1      2  16       9.49
  15    53    38      2      0  12       9.08
  16    57    44      5      1  13       8.42
  17    56    47      2      3  12       8.41
  18    55    48      1      2  15       8.64
  19    55    41      1      1  19       9.02
  20    54    40      1      2  14       9.20
  21    56    37      3      1  15

In [ ]:
# Cell purpose: verify reproducibility and disconnect MQTT cleanly.
first_run = run_simulation(params=params, ticks=tick_count, seed=seed)
second_run = run_simulation(params=params, ticks=tick_count, seed=seed)

same = [
    (
        a.tick == b.tick
        and a.sheep_count == b.sheep_count
        and a.grass_grown_cells == b.grass_grown_cells
        and a.births == b.births
        and a.deaths == b.deaths
        and a.sheep_ate_grass == b.sheep_ate_grass
        and round(a.average_energy, 8) == round(b.average_energy, 8)
    )
    for a, b in zip(first_run, second_run)
]

print(f"Deterministic run check: {all(same)}")

connector.disconnect()
print("MQTT disconnected.")

Deterministic run check: True
